In [11]:
import pickle
import pandas as pd 
from sklearn.feature_extraction import DictVectorizer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
from sklearn.pipeline import make_pipeline

In [6]:
import mlflow 

MLFLOW_TRACKING_URI = "http://0.0.0.0:5001"
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment("green-taxi-duration")


<Experiment: artifact_location='s3://mlflow-models/1', creation_time=1778362738705, experiment_id='1', last_update_time=1778362738705, lifecycle_stage='active', name='green-taxi-duration', tags={}>

In [7]:
import os
from pathlib import Path
from dotenv import load_dotenv

# 1. Load variables from .env file
# In Jupyter, use Path.cwd() instead of __file__
env_path = Path.cwd() / '.env'

# If .env is in a parent directory, walk up to find it
if not env_path.exists():
    for parent in Path.cwd().parents:
        candidate = parent / '.env'
        if candidate.exists():
            env_path = candidate
            break
    else:
        raise FileNotFoundError(f"Could not find .env file. Searched from {Path.cwd()}")

print(f"Loading .env from: {env_path}")

# Load variables
load_dotenv(dotenv_path=env_path)

# 2. Map Tigris-specific variables to MLflow/S3 standards
os.environ["MLFLOW_S3_ENDPOINT_URL"] = os.getenv("AWS_ENDPOINT_URL_S3", "")
os.environ["AWS_DEFAULT_REGION"] = os.getenv("AWS_REGION", "auto")

Loading .env from: /Users/amruthakaruturi/gitrepos/MLOps/04_deployment/web-service-mlflow/.env


In [8]:
def read_dataframe(filename):

    df = pd.read_parquet(filename, engine="fastparquet")

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)    
    return df

def prepare_dictionaries(df: pd.DataFrame):
    df["PU_DO"] = df["PULocationID"] + '_' + df["DOLocationID"]
    categorical = ["PU_DO"]
    numerical = ["trip_distance"]
    dicts = df[categorical + numerical].to_dict(orient="records")
    return dicts


In [9]:
df_train = read_dataframe('https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-01.parquet')
df_val = read_dataframe('https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-02.parquet')

target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values

dict_train = prepare_dictionaries(df_train)
dict_val = prepare_dictionaries(df_val)

In [11]:
# !pip install fastparquet

In [13]:
with mlflow.start_run():
    params = dict(max_depth = 20, n_estimators = 100, min_samples_leaf = 10, random_state=0)
    mlflow.log_params(params)

    # dv = DictVectorizer()
    # model = RandomForestRegressor(**params, n_jobs = -1)
    pipeline = make_pipeline(
        DictVectorizer(),
        RandomForestRegressor(**params, n_jobs = -1)
    )

    # X_train = dv.fit_transform(dict_train)
    # model.fit(X_train, y_train)
    pipeline.fit(dict_train, y_train)

    # X_val = dv.transform(dict_val)
    # y_pred = model.predict(X_val)
    y_pred = pipeline.predict(dict_val)
    
    rmse = root_mean_squared_error(y_pred, y_val)
    print(params, rmse)
    mlflow.log_metric("rmse", rmse)
    # mlflow.sklearn.log_model(model, artifact_path = "model")
    mlflow.sklearn.log_model(pipeline, artifact_path = "model")

    # with open('dict_vectorizer.bin', 'wb') as f_out:
    #     pickle.dump(dv, f_out)
    # mlflow.log_artifact("dict_vectorizer.bin")


{'max_depth': 20, 'n_estimators': 100, 'min_samples_leaf': 10, 'random_state': 0} 6.7558229919200725


2026/05/10 16:51:27 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
/Users/amruthakaruturi/gitrepos/MLOps/venv/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


🏃 View run sassy-ox-951 at: http://0.0.0.0:5001/#/experiments/1/runs/e0c87176278948d5949f55f67b4a0dc7
🧪 View experiment at: http://0.0.0.0:5001/#/experiments/1


In [2]:
# !pip install python-dotenv

In [1]:
env

{'ANDROID_HOME': '/Users/amruthakaruturi/Library/Android/sdk',
 'COMMAND_MODE': 'unix2003',
 'CONDA_EXE': '/Users/amruthakaruturi/anaconda3/bin/conda',
 'CONDA_PYTHON_EXE': '/Users/amruthakaruturi/anaconda3/bin/python',
 'CONDA_SHLVL': '0',
 'FPATH': '/opt/homebrew/share/zsh/site-functions:/usr/local/share/zsh/site-functions:/usr/share/zsh/site-functions:/usr/share/zsh/5.9/functions',
 'HOME': '/Users/amruthakaruturi',
 'HOMEBREW_CELLAR': '/opt/homebrew/Cellar',
 'HOMEBREW_PREFIX': '/opt/homebrew',
 'HOMEBREW_REPOSITORY': '/opt/homebrew',
 'INFOPATH': '/opt/homebrew/share/info:',
 'LANG': 'C.UTF-8',
 'LOGNAME': 'amruthakaruturi',
 'MallocNanoZone': '0',
 'NVM_BIN': '/Users/amruthakaruturi/.nvm/versions/node/v18.20.8/bin',
 'NVM_CD_FLAGS': '-q',
 'NVM_DIR': '/Users/amruthakaruturi/.nvm',
 'NVM_INC': '/Users/amruthakaruturi/.nvm/versions/node/v18.20.8/include/node',
 'ORIGINAL_XDG_CURRENT_DESKTOP': 'undefined',
 'OSLogRateLimit': '64',
 'PATH': '/Users/amruthakaruturi/gitrepos/MLOps/venv